# Customer Transaction Segmentation With Rfm

**AI Services MS2 | Supplementary Practice D | 60 Marks | Exactly 20 Evaluation Checks**

**Instructor reference notebook — complete working code.**

**Business task.** An online retailer wants to understand customer purchasing patterns without preassigned labels. The organization records one row per invoice line, including customer, invoice reference, transaction date, quantity and unit price. A customer may appear in several invoices, each containing several line items.

**Objective.** Build an unsupervised customer-segmentation notebook. Clean invoice lines, construct RFM features, compare 3/4/5-cluster K-Means pipelines using silhouette and Davies-Bouldin scores, choose the best candidate, and assign cluster IDs to new customers without refitting.

## 1. Imports, parameters and feature groups

In [ ]:
import joblib

import numpy as np

import pandas as pd

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler, FunctionTransformer

from sklearn.cluster import KMeans

from sklearn.metrics import silhouette_score, davies_bouldin_score

REFERENCE_DATE = pd.Timestamp("2026-12-01")

RFM_COLUMNS = ["recency_days", "frequency", "monetary"]

## Implementation contract — independently testable functions

**1. `load_data(filename: str) -> pd.DataFrame`**  
**Returns:** Return customer_id, recency_days, frequency, monetary in that exact order; one row per customer.  
• Read invoice-line CSV, strip customer_id and invoice_no, remove a trailing .0 from customer IDs, parse invoice_date with errors="coerce".  
• Convert quantity and unit_price to numeric and replace infinities with NaN. Remove missing/blank key values and cancel invoices whose invoice_no starts with C in any letter case.  
• Keep quantity > 0, unit_price > 0 and invoice_date strictly before the fixed reference date 2026-12-01. Compute amount=quantity×unit_price.  
• Group by customer_id in first-appearance order; obtain last_purchase=max(invoice_date), frequency=nunique(invoice_no), monetary=sum(amount). Derive recency_days from the fixed reference date to the normalized last_purchase date.

**2. `build_pipeline(n_clusters: int) -> Pipeline`**  
**Returns:** Return an unfitted four-step numeric K-Means pipeline.  
• Use SimpleImputer(strategy="median"), FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler(), then KMeans(n_clusters=n_clusters, random_state=42, n_init=10).

**3. `prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame]`**  
**Returns:** Return (train_customers, test_customers), two cleaned customer-level DataFrames.  
• Apply the common RFM loader to both CSVs. The test file contains independent customers; no target label or y Series is used.

**4. `train_models(train_customers: pd.DataFrame) -> dict[str, Pipeline]`**  
**Returns:** Return three independently fitted pipelines keyed kmeans_3, kmeans_4, kmeans_5.  
• Train K-Means with k=3, k=4, k=5 using only the three numeric RFM features.

**5. `evaluate_models(trained_models: dict[str, Pipeline], test_customers: pd.DataFrame) -> pd.DataFrame`**  
**Returns:** Return ranked comparison with model_name, silhouette_score, davies_bouldin_score.  
• Predict a cluster label for each held-back customer. Apply the fitted preprocessing through pipeline[:-1].transform(X_test) to obtain values in the same feature space used by K-Means.  
• Calculate silhouette_score and davies_bouldin_score on the transformed values and predicted labels if at least two clusters exist and fewer clusters than samples. Otherwise silhouette=-1.0 and Davies-Bouldin=NaN.  
• Sort silhouette descending, Davies-Bouldin ascending, model_name ascending; NaN metrics go last. No refitting.

**6. `select_best_model(comparison: pd.DataFrame, trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]`**  
**Returns:** Return the first-ranked name and original fitted pipeline.  
• The comparison has already been ranked. Do not train another model or instantiate a replacement.

**7. `segment_customers(model: Pipeline, customers: pd.DataFrame) -> pd.DataFrame`**  
**Returns:** Return customer_id and integer segment_id in that exact order.  
• Predict clusters using the saved K-Means pipeline on exactly the three RFM fields; preserve customer order. Cluster numbers are arbitrary labels, not ordered quality scores.

In [ ]:
def load_data(filename: str) -> pd.DataFrame:
    """Clean raw invoice lines and aggregate to one RFM row per customer."""
    df = pd.read_csv(filename, dtype={"invoice_no": "string", "customer_id": "string"})
    df["customer_id"] = (df["customer_id"].str.strip()
                        .str.replace(r"\.0$", "", regex=True))
    df["invoice_no"] = df["invoice_no"].str.strip()
    df["invoice_date"] = pd.to_datetime(df["invoice_date"], format="mixed", dayfirst=True, errors="coerce")
    for col in ["quantity", "unit_price"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["customer_id", "invoice_no", "invoice_date", "quantity", "unit_price"])
    df = df[(df["customer_id"] != "") & (df["invoice_no"] != "")]
    df = df[~df["invoice_no"].str.upper().str.startswith("C")]
    df = df[(df["quantity"] > 0) & (df["unit_price"] > 0) & (df["invoice_date"] < REFERENCE_DATE)].copy()
    df["amount"] = df["quantity"] * df["unit_price"]
    customers = df.groupby("customer_id", sort=False).agg(
        last_purchase=("invoice_date", "max"),
        frequency=("invoice_no", "nunique"),
        monetary=("amount", "sum")
    ).reset_index()
    customers["recency_days"] = (REFERENCE_DATE - customers["last_purchase"].dt.normalize()).dt.days
    return customers[["customer_id", *RFM_COLUMNS]].reset_index(drop=True)

def build_pipeline(n_clusters: int) -> Pipeline:
    """Numeric-only imputation -> log1p -> scaling -> KMeans clustering pipeline."""
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scaler", StandardScaler()),
        ("model", KMeans(n_clusters=n_clusters, random_state=42, n_init=10))
    ])

def prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return cleaned customer-level RFM DataFrames without target labels."""
    return load_data(train_file), load_data(test_file)

def train_models(train_customers: pd.DataFrame) -> dict[str, Pipeline]:
    """Train KMeans pipelines with 3, 4 and 5 clusters."""
    models = {"kmeans_3": 3, "kmeans_4": 4, "kmeans_5": 5}
    trained_models = {}
    for name, n_clusters in models.items():
        pipeline = build_pipeline(n_clusters)
        pipeline.fit(train_customers[RFM_COLUMNS])
        trained_models[name] = pipeline
    return trained_models

def evaluate_models(trained_models: dict[str, Pipeline],
                    test_customers: pd.DataFrame) -> pd.DataFrame:
    """Rank candidate KMeans models by silhouette descending then Davies-Bouldin ascending."""
    X_test = test_customers[RFM_COLUMNS]
    results = []
    for name, pipeline in trained_models.items():
        labels = pipeline.predict(X_test)
        transformed = pipeline[:-1].transform(X_test)
        valid = 2 <= len(np.unique(labels)) < len(labels)
        results.append({
            "model_name": name,
            "silhouette_score": silhouette_score(transformed, labels) if valid else -1.0,
            "davies_bouldin_score": davies_bouldin_score(transformed, labels) if valid else np.nan
        })
    return pd.DataFrame(results).sort_values(
        ["silhouette_score", "davies_bouldin_score", "model_name"],
        ascending=[False, True, True], na_position="last"
    ).reset_index(drop=True)

def select_best_model(comparison: pd.DataFrame,
                      trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]:
    """Return the top-ranked model name and its already fitted pipeline."""
    name = str(comparison.iloc[0]["model_name"])
    return name, trained_models[name]

def segment_customers(model: Pipeline, customers: pd.DataFrame) -> pd.DataFrame:
    """Assign segment IDs to customer RFM rows while preserving customer order."""
    labels = model.predict(customers[RFM_COLUMNS]).astype(int)
    return pd.DataFrame({"customer_id": customers["customer_id"].to_numpy(),
                         "segment_id": labels})

## Training and saved-model prediction

The training and prediction workflows are intentionally separate. Training fits and persists the chosen estimator with its preprocessing. Prediction must load the saved model and process new rows without calling `fit()`.

**`train() -> tuple[str, pd.DataFrame]`** — Return selected cluster-model name and comparison. Fit and compare candidate clusters; persist best_model.pkl and model_comparison.csv.

**`predict() -> pd.DataFrame`** — Return segment assignments for the prediction CSV. Load saved model, aggregate retail_predict.csv to customer level using the common loader, and write retail_segments.csv without fitting.

**Required output files:** `best_model.pkl`, `model_comparison.csv`, `retail_segments.csv`

In [ ]:
def train() -> tuple[str, pd.DataFrame]:
    """Fit clusters, save best_model.pkl and model_comparison.csv."""
    train_customers, test_customers = prepare_datasets("retail_train.csv", "retail_test.csv")
    trained = train_models(train_customers)
    comparison = evaluate_models(trained, test_customers)
    name, model = select_best_model(comparison, trained)
    joblib.dump(model, "best_model.pkl")
    comparison.to_csv("model_comparison.csv", index=False)
    return name, comparison

def predict() -> pd.DataFrame:
    """Reload best_model.pkl and segment queued transactions without refitting."""
    customers = load_data("retail_predict.csv")
    model = joblib.load("best_model.pkl")
    result = segment_customers(model, customers)
    result.to_csv("retail_segments.csv", index=False)
    return result

## 4. Run the workflow

Execute the **training** call first, then the **prediction** call. You can also run these in separate notebook sessions after a model has been saved.

In [ ]:
best_name, comparison = train()
display(comparison)

In [ ]:
result = predict()
display(result.head())